In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

mariamhany44_441_preprocessed_with_features_path = kagglehub.dataset_download('mariamhany44/441-preprocessed-with-features')

print('Data source import complete.')


100%|██████████| 4.73G/4.73G [04:52<00:00, 17.4MB/s]

Extracting files...


Data source import complete.


In [3]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    classification_report
)
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pathlib import Path
import pandas as pd
import random
import torch
import os

torch.set_num_threads(4)
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

# =========================
# CONFIG
# =========================
DATA_DIR = os.path.join(
    mariamhany44_441_preprocessed_with_features_path,
    "final_dataset"
)
OUTPUT_DIR = "/content/model_results_2"
CLASS_MAP_PATH = os.path.join(
    DATA_DIR,
    "label_encoder.npy"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
LR = 0.0025
WEIGHT_DECAY = 0.001
LABEL_SMOOTH = 0.12
EPOCHS = 100
PATIENCE = 15
GRAD_CLIP = 1.0
CHANNELS = 256
DROPOUT = 0.30
SEED = 42

# =========================
# DETERMINISM
# =========================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# =========================
# FEATURE GROUPS
# =========================
feature_slices = {
    "v2": (1314, 1752),
    "pose_bones": (2190, 2208),
    "lh_bones": (2208, 2268),
    "rh_bones": (2268, 2328),
    "lh_angles": (2328, 2343),
    "rh_angles": (2343, 2358),
    "relative": (2358, 2673),
    "distances": (2673, 2676),
    "handshape": (2676, 2680)
}


# =========================
# DATASET
# =========================
class NPZDataset(Dataset):
    def __init__(self, folder, remove_groups=None):
        self.files = sorted(list(Path(folder).glob("*.npz")))
        self.remove_groups = remove_groups or []

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])

        x = data["x"].astype(np.float32)
        y = int(data["y"])
        m = data["mask"].astype(np.float32)

        for g in self.remove_groups:
            s, e = feature_slices[g]
            x[:, s:e] = 0.0

        return torch.from_numpy(x), torch.from_numpy(m), y

# =========================
# MODEL
# =========================
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.Dropout(0.3),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.Dropout(0.3),
            nn.ReLU(inplace=True),
        )
        self.res = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        y = self.net(x)
        if y.size(2) != x.size(2):
            y = y[..., :x.size(2)]
        return y + self.res(x)

class TCN(nn.Module):
    def __init__(self, input_dim, num_classes,
                 channels=CHANNELS,
                 dropout=DROPOUT):
        super().__init__()

        layers = []
        for i in range(4):
            in_dim = input_dim if i == 0 else channels
            layers.append(
                TemporalBlock(
                    in_dim,
                    channels,
                    dilation=2**i
                )
            )

        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(channels, num_classes)
        self.dropout = nn.Dropout(dropout)

    def masked_pool(self, x, mask):
        mask = mask.unsqueeze(1)
        x = x * mask
        summed = x.sum(dim=2)
        valid = mask.sum(dim=2).clamp(min=1)
        return summed / valid

    def forward(self, x, mask):
        x = x.transpose(1, 2)
        x = self.tcn(x)
        x = self.masked_pool(x, mask)
        x = self.dropout(x)
        return self.fc(x)

# =========================
# RUN EXPERIMENT
# =========================
def run_experiment(name, remove_groups):

    set_seed(SEED)

    trial_dir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(trial_dir, exist_ok=True)

    print(name)


    train_ds = NPZDataset(os.path.join(DATA_DIR, "train"), remove_groups)
    val_ds   = NPZDataset(os.path.join(DATA_DIR, "val"), remove_groups)
    test_ds  = NPZDataset(os.path.join(DATA_DIR, "test"), remove_groups)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

    sample_x, _, _ = train_ds[0]
    FEATURE_DIM = sample_x.shape[1]

    labels = [train_ds[i][2] for i in range(len(train_ds))]
    unique_labels = sorted(set(labels))
    num_classes = len(unique_labels)
    loaded_map = np.load(CLASS_MAP_PATH, allow_pickle=True).item()

    # invert map: gloss -> id  becomes  id -> gloss
    class_map = {v: k for k, v in loaded_map.items()}

    target_names = [class_map[i] for i in unique_labels]
    # =========================
    # DATA STATS PRINTING
    # =========================
    print("\n📊 DATASET STATS")
    print(f"Train samples: {len(train_ds)}")
    print(f"Val samples:   {len(val_ds)}")
    print(f"Test samples:  {len(test_ds)}")

    print(f"Total samples: {len(train_ds) + len(val_ds) + len(test_ds)}")

    print(f"\nNumber of classes: {num_classes}")

    print(f"\nFeature dimension: {FEATURE_DIM}")


    print(DATA_DIR)
    print(os.listdir(DATA_DIR))

    print(len(list(Path(os.path.join(DATA_DIR, "train")).glob("*.npz"))))
    print(len(list(Path(os.path.join(DATA_DIR, "val")).glob("*.npz"))))
    print(len(list(Path(os.path.join(DATA_DIR, "test")).glob("*.npz"))))

    model = TCN(
        FEATURE_DIM,
        num_classes,
        channels=CHANNELS,
        dropout=DROPOUT
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    best_val = float("inf")
    patience = 0

    logs = []

    for epoch in range(EPOCHS):

        model.train()
        tl, tc, tt = 0, 0, 0

        for x, m, y in train_loader:
            x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            out = model(x, m)
            loss = criterion(out, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            tl += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item()
            tt += y.size(0)

        train_loss = tl / tt
        train_acc = tc / tt

        model.eval()
        vl, vc, vt = 0, 0, 0

        with torch.no_grad():
            for x, m, y in val_loader:
                x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
                out = model(x, m)
                loss = criterion(out, y)

                vl += loss.item() * y.size(0)
                vc += (out.argmax(1) == y).sum().item()
                vt += y.size(0)

        val_loss = vl / vt
        val_acc = vc / vt

        scheduler.step(val_loss)

        print(f"E{epoch+1:02d} | TL {train_loss:.4f} | TA {train_acc:.4f} | VL {val_loss:.4f} | VA {val_acc:.4f}")

        logs.append({
            "epoch": epoch+1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        if val_loss < best_val:
            best_val = val_loss
            patience = 0
            torch.save(model.state_dict(), os.path.join(trial_dir, "best_model.pth"))
        else:
            patience += 1
            if patience >= PATIENCE:
                print("Early stopping")
                break

    pd.DataFrame(logs).to_csv(os.path.join(trial_dir, "epoch_logs.csv"), index=False)

    # TEST
    model.load_state_dict(torch.load(os.path.join(trial_dir, "best_model.pth")))
    model.eval()

    tl, tc, tt = 0, 0, 0
    preds, targets = [], []

    with torch.no_grad():
        for x, m, y in test_loader:
            x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
            out = model(x, m)
            loss = criterion(out, y)

            tl += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item()
            tt += y.size(0)

            preds.extend(out.argmax(1).cpu().numpy())
            targets.extend(y.cpu().numpy())

    test_loss = tl / tt
    test_acc = tc / tt

    print("Test accuracy:", test_acc)
    print("Test loss:", test_loss)

    # =========================
    # PER-CLASS METRICS
    # =========================
    report = classification_report(
        targets,
        preds,
        labels=unique_labels,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).transpose()

    report_df.to_csv(
        os.path.join(trial_dir, "classification_report.csv")
    )

    print("\nPer-class metrics saved.")
    wrong = []

    for p, t in zip(preds, targets):
        if p != t:
            wrong.append({
                "pred_id": int(p),
                "pred_gloss": class_map[int(p)],

                "true_id": int(t),
                "true_gloss": class_map[int(t)]
            })
    pd.DataFrame(wrong).to_csv(os.path.join(trial_dir, "wrong_predictions.csv"), index=False)
    print("-----------------------------------------------------------------")
    return {
        "experiment": name,
        "removed": ",".join(remove_groups),
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc
    }

# =========================
# MAIN
# =========================
if __name__ == "__main__":

    result = run_experiment(
        name="TCN_model",
        remove_groups=[]
    )

    pd.DataFrame([result]).to_csv(
        os.path.join(OUTPUT_DIR, "results.csv"),
        index=False
    )

    print("\nTraining completed and saved.")

TCN_model

📊 DATASET STATS
Train samples: 37485
Val samples:   2261
Test samples:  2395
Total samples: 42141

Number of classes: 441

Feature dimension: 928
/root/.cache/kagglehub/datasets/mariamhany44/441-preprocessed-with-features/versions/1/final_dataset
['train', 'test', 'val', 'label_encoder.npy']
37485
2261
2395
E01 | TL 4.0786 | TA 0.2682 | VL 3.1299 | VA 0.4759
E02 | TL 2.4530 | TA 0.6790 | VL 2.3757 | VA 0.6908
E03 | TL 2.0239 | TA 0.8124 | VL 2.1766 | VA 0.7598
E04 | TL 1.8528 | TA 0.8682 | VL 2.0638 | VA 0.7753
E05 | TL 1.7464 | TA 0.8988 | VL 2.0483 | VA 0.7873
E06 | TL 1.6801 | TA 0.9184 | VL 2.0481 | VA 0.7780
E07 | TL 1.6309 | TA 0.9337 | VL 1.9609 | VA 0.8129
E08 | TL 1.5953 | TA 0.9432 | VL 1.9576 | VA 0.8195
E09 | TL 1.5630 | TA 0.9516 | VL 2.0065 | VA 0.7988
E10 | TL 1.5385 | TA 0.9588 | VL 1.8993 | VA 0.8297
E11 | TL 1.5248 | TA 0.9616 | VL 1.9038 | VA 0.8275
E12 | TL 1.5107 | TA 0.9646 | VL 1.8771 | VA 0.8386
E13 | TL 1.4925 | TA 0.9708 | VL 1.8851 | VA 0.8359
E14 